## Make the W operator

## Notes:
- There are two coordinate systems; one on the images and one on the scene. The images are in the (w, z) system and the scene is in the (x, y) system.
## Bugs:
- Nothing yet!

## To-do:
- WRITE HELLA COMMENTS ON THE CODE
- Discuss what the units are! Check the overleaf to confirm.
- Make the operator that returns bessel evaluations at the correct locations for some list of Js
- Make _J_zeros
- Make the Js mutable or pythonic or whatever
- Figure out normalization of W
- Figure out correct color-scheme
- Start writing appendix A about the W operator

# import numpy as np
import pylab as plt
import os
import scipy.special as special
from astropy.table import Table
from astropy.io import fits

In [ ]:
WPIXSCALE = 0.059 # arcsec
ZPIXSCALE = 3. * WPIXSCALE
N, Nw, Nz = 120, 18, 12
R = ZPIXSCALE * Nz / 2 + WPIXSCALE + 0.01

In [ ]:
ws, zs = (WPIXSCALE / 2) * (np.arange(2 * Nw) - 0.5 * (2 * Nw - 1.)), (WPIXSCALE / 2) * (np.arange(6 * Nz) - 0.5 * (6 * Nz - 1.))
zgrid, wgrid = np.meshgrid(zs, ws)
rgrid_wz = np.sqrt(wgrid**2 + zgrid**2)
phigrid_wz = np.atan2(zgrid, wgrid)

In [ ]:
def get_pixel_coords(w, z):
    return np.array([w / WPIXSCALE, z / ZPIXSCALE])

In [ ]:
def get_pixel_average(arr):
    meanz = np.mean(arr.reshape((2 * Nw, Nz, 6, np.int32(arr.size / (12 * Nz * Nw)))), 2)
    return np.mean(meanz.reshape((Nw, 2, Nz, np.int32(meanz.size / (2 * Nz * Nw)))), 1)

In [ ]:
plt.figure(figsize=(27, 9))
wzpixcoords = get_pixel_coords(wgrid, zgrid)
plt.scatter(wzpixcoords[0], wzpixcoords[1], s=0.3, c='k')
plt.vlines(np.arange(Nw+1) - 0.5*Nw, ymin=0-0.5*Nz, ymax=0.5*Nz, linewidth=0.5, color='k')
plt.hlines(np.arange(Nz+1) - 0.5*Nz, xmin=0-0.5*Nw, xmax=0.5*Nw, linewidth=0.5, color='k')
plt.gca().set_aspect(3.)

In [ ]:
wavg = get_pixel_average(wgrid)
zavg = get_pixel_average(zgrid)
wzavgpixcoords = get_pixel_coords(wavg, zavg)
plt.figure(figsize=(27, 9))
plt.scatter(wzavgpixcoords[0], wzavgpixcoords[1], s=0.3, c='k')
plt.vlines(np.arange(Nw+1) - 0.5*Nw, ymin=0-0.5*Nz, ymax=0.5*Nz, linewidth=0.5, color='k')
plt.hlines(np.arange(Nz+1) - 0.5*Nz, xmin=0-0.5*Nw, xmax=0.5*Nw, linewidth=0.5, color='k')
plt.gca().set_aspect(3.)

In [ ]:
def Jzeros(Nm, Nn):
    Jzeros = np.zeros([2 * Nm + 1, Nn])
    for m in np.arange(Nm):
        col = special.jn_zeros(m, Nn)
        Jzeros[Nm + m] = col
        Jzeros[Nm - m] = col
    return Jzeros

def besselize(Nm, Nn, rs, phis, R):
    ms_rav = np.ravel(np.ones(Nn)[None,:] * (np.arange(2 * Nm + 1)-Nm)[:,None])
    Jzeros_rav = np.ravel(Jzeros(Nm, Nn))
    samples = special.jv(np.abs(ms_rav[None,:]), Jzeros_rav[None,:] * rs[:,:,None] / R) * np.exp(1j * ms_rav[None,:] * phis[:,:,None])
    return samples

def W(Nm, Nn, rs, phis, R):
    samples = besselize(Nm, Nn, rs, phis, R)
    pixs = get_pixel_average(samples)
    W_fft = np.fft.fft2(pixs, axes=(0, 1))
    return W_fft

In [ ]:
Nm = 70
Nn = 70

W_test = W(Nm, Nn, rgrid_wz, phigrid_wz, R)

In [ ]:
fig, axs = plt.subplots(11, 10, figsize=(12, 24))
cscale = 3 / np.sqrt(11*10)
for n in range(10):
    for m in range(11):
        a_mns = np.zeros([2*Nm+1, Nn])
        a_mns[Nm - 5 + m, n] = 1
        a_mns = np.ravel(a_mns)
        pixtest = np.fft.ifft2(W_test@a_mns)
        axs[m][n].imshow(np.real(pixtest.T), origin="lower", interpolation="nearest", cmap="RdBu", aspect=3, vmin=-cscale, vmax=cscale)
        axs[m][n].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(11, 10, figsize=(12, 24))
cscale = 3 / np.sqrt(11*10)
for n in range(10):
    for m in range(11):
        a_mns = np.zeros([2*Nm+1, Nn])
        a_mns[Nm - 5 + m, n] = 1
        a_mns = np.ravel(a_mns)
        pixtest = np.fft.ifft2(W_test@a_mns)
        axs[m][n].imshow(np.imag(pixtest.T), origin="lower", interpolation="nearest", cmap="RdBu", aspect=3., vmin=-cscale, vmax=cscale)
        axs[m][n].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
xs = (WPIXSCALE / 2) * (np.arange(4 + np.int32(2 * R / (WPIXSCALE / 2))) - 0.5 * (4 + np.int32(2 * R / (WPIXSCALE / 2)) - 1.))
ygrid, xgrid = np.meshgrid(xs, xs)
rgrid_sc = np.sqrt(xgrid**2 + ygrid**2)
circle_mask = rgrid_sc > R
phigrid_sc = np.atan2(ygrid, xgrid)

def make_circle(arr):
    arr[circle_mask] = np.nan
    return arr

plt.figure(figsize=(8, 8))
angles_test = 2*np.pi*np.arange(1e6) / 1e6
plt.plot(R*np.cos(angles_test), R*np.sin(angles_test), linewidth=0.4, c='k')
plt.vlines(0, ymin=-R, ymax=R, linewidth=0.4, color='k')
plt.hlines(0, xmin=-R, xmax=R, linewidth=0.4, color='k')
plt.scatter(make_circle(xgrid), make_circle(ygrid), s=0.3, c='k')
plt.gca().set_aspect(1.)

In [ ]:
image_op = besselize(Nm, Nn, rgrid_sc, phigrid_sc, R)

In [ ]:
fig, axs = plt.subplots(Nw, Nz, figsize=(12, 18))
cscale = 1300 / np.sqrt(np.pi * R**2 / (WPIXSCALE / 2)**2)
for k in range(Nw):
    for l in range(Nz):
        y_kls = np.zeros([Nw, Nz])
        y_kls[k, l] = 1
        y_kls = np.ravel(y_kls)
        imtest = make_circle(np.real(image_op@((y_kls.T@(W_test.reshape(216, 9870))).T)).T)
        axs[k][l].imshow(imtest, origin="lower", interpolation="nearest", cmap="RdBu", vmax=cscale, vmin=-cscale)
        axs[k][l].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(Nw, Nz, figsize=(12, 18))
cscale = 1300 / np.sqrt(np.pi * R**2 / (WPIXSCALE / 2)**2)
for k in range(Nw):
    for l in range(Nz):
        y_kls = np.zeros([Nw, Nz])
        y_kls[k, l] = 1
        y_kls = np.ravel(y_kls)
        imtest = np.imag(image_op@((y_kls.T@(W_test.reshape(216, 9870))).T)).T
        imtest[circle_mask] = np.nan
        axs[k][l].imshow(imtest, origin="lower", interpolation="nearest", cmap="RdBu", vmax=cscale, vmin=-cscale)
        axs[k][l].axis("off")
plt.tight_layout()
plt.show()